# Økonometri Aflevering
## Opsætning og biblioteker

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import scipy.stats as stats
import matplotlib.pyplot as plt
from linearmodels.iv import IV2SLS

## Hjælpefunktioner
Her defineres funktioner til beregning af OLS-estimater og F-tests, som benyttes på tværs af opgaverne.

In [2]:
def OLS_ren(X, y, navne):
    n = X.shape[0]
    k = X.shape[1] - 1

    X_T_X_inv = np.linalg.inv(X.T @ X)
    betahat = X_T_X_inv @ X.T @ y
    
    y_hat = X @ betahat
    u_hat = y - y_hat
    y_bar = np.mean(y)

    SST = np.sum((y - y_bar)**2)
    SSR = np.sum(u_hat**2)
    SSE = np.sum((y_hat - y_bar)**2)
    R2 = SSE / SST
    
    resvar = (1 / (n - k - 1)) * np.dot(u_hat, u_hat)
    
    var_betahatX = resvar * X_T_X_inv
    se_betahat = np.sqrt(np.diagonal(var_betahatX))
    
    t_stat = betahat / se_betahat
    p_values = 2 * stats.t.sf(np.abs(t_stat), n - k - 1)

    print("==============================================================================")
    print(f"Number of Observations: {n}")
    print(f"Residual Degrees of Freedom: {n - k - 1}")
    print(f"R-squared: {R2:.4f}")
    print(f"SST: {SST:.4f}, SSE: {SSE:.4f}, SSR: {SSR:.4f}")
    print("==============================================================================")
    
    results = pd.DataFrame({
        'Koefficient': betahat,
        'Std.fejl': se_betahat,
        't-værdi': t_stat,
        'P>|t|': p_values
    }, index=navne)
    
    formatted_results = results.copy()
    for col in formatted_results.columns:
        formatted_results[col] = formatted_results[col].apply(lambda x: f"{x:.4f}")
    
    display(formatted_results)

def F_test(SSR_r, SSR_ur, q, nk1):
    F = ((SSR_r - SSR_ur) / q) / (SSR_ur / nk1)
    crit_lim = float(stats.f.ppf(0.95, dfn=q, dfd=nk1))
    p_val = float(1 - stats.f.cdf(F, dfn=q, dfd=nk1))
    print(f"F-værdi: {F:.4f}")
    print(f"Kritisk grænse (5%): {crit_lim:.4f}")
    print(f"P-værdi (5%): {p_val:.4f}")

## Opgave 4.c

In [7]:
n = 500
def simuler(n, gamma1=1, gamma2=1):

    # Step 1. Definer parameterværdier
    beta0 = 0.5
    beta1 = -0.1
    gamma0 = 0.25

    # Step 2. Simular data
    z_i = np.random.normal(loc = 0 , scale= 1, size = n)
    xi_i = np.random.normal(loc = 0,  scale= 1, size = n)
    nu_i = np.random.normal(loc = 0, scale= 1, size = n)
    eta_0 = (gamma0-beta0) / (beta1- gamma1)
    eta_1 = gamma2 / (beta1-gamma1)
    u_i = (nu_i - xi_i) / (beta1-gamma1)
    p_i = eta_0 + eta_1 * z_i + u_i
    y = beta0 + beta1*p_i + xi_i

    X_ols = sm.add_constant(p_i)
    ols_res = sm.OLS(y, X_ols).fit()
    beta1_OLS = np.array(ols_res.params)[1]

    # Step 4. Estimer med IV (z_i som instrument for p_i)
    const = np.ones(n) # IV2SLS foretrækker en eksplicit konstant-array
    iv_res = IV2SLS(dependent=y, exog=const, endog=p_i, instruments=z_i).fit()
    beta1_IV = np.array(iv_res.params)[1]

    return beta1_OLS, beta1_IV

def monte_carlo (reps=500, gamma1=1, gamma2=1):
    np.random.seed(0)

    beta1_ols_results = []
    beta1_iv_results = []

    for rep in range(reps):
        b_ols, b_iv = simuler(n, gamma1=gamma1, gamma2=gamma2)
        beta1_ols_results.append(b_ols)
        beta1_iv_results.append(b_iv)
        
    # Beregn bias (gennemsnitligt estimat - sand værdi)
    true_beta1 = -0.1
    bias_ols = np.mean(beta1_ols_results) - true_beta1
    bias_iv = np.mean(beta1_iv_results) - true_beta1
    
    # Beregn varians af estimaterne
    var_ols = np.var(beta1_ols_results, ddof=1)
    var_iv = np.var(beta1_iv_results, ddof=1)
    
    return bias_ols, var_ols, bias_iv, var_iv

print("Scenarie 1 (Baseline):")
bias_ols1, var_ols1, bias_iv1, var_iv1 = monte_carlo(reps=500, gamma1=1, gamma2=1)
print(f"OLS - Bias: {bias_ols1:.5f} | Varians: {var_ols1:.5f}")
print(f"IV  - Bias: {bias_iv1:.5f} | Varians: {var_iv1:.5f}\n")

print("Scenarie 2 (gamma2 = 0.25):")
bias_ols2, var_ols2, bias_iv2, var_iv2 = monte_carlo(reps=500, gamma1=1, gamma2=0.25)
print(f"OLS - Bias: {bias_ols2:.5f} | Varians: {var_ols2:.5f}")
print(f"IV  - Bias: {bias_iv2:.5f} | Varians: {var_iv2:.5f}\n")

print("Scenarie 3 (gamma1 = 0.25):")
bias_ols3, var_ols3, bias_iv3, var_iv3 = monte_carlo(reps=500, gamma1=0.25, gamma2=1)
print(f"OLS - Bias: {bias_ols3:.5f} | Varians: {var_ols3:.5f}")
print(f"IV  - Bias: {bias_iv3:.5f} | Varians: {var_iv3:.5f}")

Scenarie 1 (Baseline):
OLS - Bias: 0.36753 | Varians: 0.00056
IV  - Bias: -0.00030 | Varians: 0.00236

Scenarie 2 (gamma2 = 0.25):
OLS - Bias: 0.53391 | Varians: 0.00064
IV  - Bias: -0.03065 | Varians: 0.05343

Scenarie 3 (gamma1 = 0.25):
OLS - Bias: 0.11694 | Varians: 0.00006
IV  - Bias: -0.00010 | Varians: 0.00024
